# Bayesian Model Comparison: The Evidence Framework

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/bayesian_model_comparison.ipynb)

Companion notebook to the blog post [Bayesian Model Comparison: The Evidence Framework](https://sesen.ai/blog/bayesian-model-comparison-evidence-framework).

We reproduce Bishop's polynomial regression evidence plot (PRML Fig 3.14), then compare the Bayesian log marginal likelihood with BIC, AIC, and held-out test MSE on the same dataset. The goal is to see Occam's razor emerging from the evidence integral, and to watch AIC and BIC break in the interpolation regime where the Bayesian approach stays sane.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.set_printoptions(precision=3, suppress=True)

## 1. The dataset: Bishop's sinusoidal toy

Ten noisy observations of $\sin(2\pi x)$, same setup as Bishop PRML §1.1.

In [ ]:
def true_function(x):
    return np.sin(2 * np.pi * x)


def sample_toy_data(N=10, noise_sigma=0.3, seed=42):
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(0, 1, N))
    t = true_function(x) + rng.normal(0, noise_sigma, N)
    return x, t


x, t = sample_toy_data(N=10, noise_sigma=0.3, seed=42)
x_test, t_test = sample_toy_data(N=100, noise_sigma=0.3, seed=7)

fig, ax = plt.subplots(figsize=(7, 4))
xs = np.linspace(0, 1, 300)
ax.plot(xs, true_function(xs), "k--", label="sin(2\u03c0x)")
ax.scatter(x, t, color="#D4A24C", edgecolor="#1B2D3D", s=55, zorder=5, label="observed")
ax.set_xlabel("x"); ax.set_ylabel("t"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 2. Closed-form Bayesian linear regression

With a Gaussian prior $p(\mathbf{w}) = \mathcal{N}(\mathbf{0}, \alpha^{-1} \mathbf{I})$ and Gaussian likelihood with precision $\beta$, the posterior over weights is also Gaussian (Bishop eqs. 3.53–3.54):

$$
\mathbf{S}_N^{-1} = \alpha \mathbf{I} + \beta \mathbf{\Phi}^\top \mathbf{\Phi}, \qquad \mathbf{m}_N = \beta \mathbf{S}_N \mathbf{\Phi}^\top \mathbf{t}.
$$

In [ ]:
def phi_poly(x, M):
    x = np.asarray(x).ravel()
    return np.stack([x ** k for k in range(M + 1)], axis=1)


def bayesian_linear_regression(Phi, t, alpha, beta):
    M_plus = Phi.shape[1]
    A = alpha * np.eye(M_plus) + beta * Phi.T @ Phi  # S_N^-1
    S_N = np.linalg.inv(A)
    m_N = beta * S_N @ Phi.T @ t
    return m_N, S_N

## 3. The log marginal likelihood

Bishop's closed-form expression (eq. 3.86):

$$
\ln p(\mathbf{t} \mid \alpha, \beta) = \tfrac{M}{2} \ln \alpha + \tfrac{N}{2} \ln \beta - E(\mathbf{m}_N) - \tfrac{1}{2} \ln |\mathbf{A}| - \tfrac{N}{2} \ln(2\pi),
$$

with $E(\mathbf{m}_N) = \tfrac{\beta}{2} \|\mathbf{t} - \mathbf{\Phi}\mathbf{m}_N\|^2 + \tfrac{\alpha}{2}\mathbf{m}_N^\top \mathbf{m}_N$.

We also split the evidence into a data-fit term and an Occam penalty term.

In [ ]:
def log_evidence(Phi, t, alpha, beta):
    N = len(t)
    M_plus = Phi.shape[1]
    m_N, S_N = bayesian_linear_regression(Phi, t, alpha, beta)
    A = alpha * np.eye(M_plus) + beta * Phi.T @ Phi
    _, logdet_A = np.linalg.slogdet(A)
    residual = t - Phi @ m_N
    E_mN = 0.5 * beta * residual @ residual + 0.5 * alpha * m_N @ m_N
    log_ev = (
        0.5 * M_plus * np.log(alpha)
        + 0.5 * N * np.log(beta)
        - E_mN
        - 0.5 * logdet_A
        - 0.5 * N * np.log(2 * np.pi)
    )
    data_fit = -0.5 * beta * (residual @ residual) + 0.5 * N * np.log(beta / (2 * np.pi))
    complexity = log_ev - data_fit
    return float(log_ev), float(data_fit), float(complexity)

## 4. AIC and BIC as asymptotic approximations

Both criteria plug the MLE point estimate into $-2 \ln L_\max$ and add a complexity penalty. AIC penalises each parameter by $2$, BIC by $\ln N$.

In [ ]:
def information_criteria(Phi, t):
    w_ml, *_ = np.linalg.lstsq(Phi, t, rcond=None)
    residual = t - Phi @ w_ml
    N = len(t)
    sigma2_ml = max((residual @ residual) / N, 1e-12)
    log_lik = -0.5 * N * np.log(2 * np.pi * sigma2_ml) - 0.5 * N
    k = Phi.shape[1] + 1  # weights + sigma^2
    aic = -2 * log_lik + 2 * k
    bic = -2 * log_lik + k * np.log(N)
    return float(aic), float(bic), float(log_lik)

## 5. Scan across polynomial orders $M = 0, \ldots, 9$

In [ ]:
alpha = 5e-3
beta = 1.0 / 0.3 ** 2
orders = list(range(10))

log_evs, data_fits, complexities = [], [], []
aics, bics, log_liks = [], [], []
train_mses, test_mses = [], []
fit_curves = []
x_plot = np.linspace(0, 1, 300)

for M in orders:
    Phi = phi_poly(x, M)
    Phi_test = phi_poly(x_test, M)
    Phi_plot = phi_poly(x_plot, M)
    le, df, cp = log_evidence(Phi, t, alpha, beta)
    aic, bic, ll = information_criteria(Phi, t)
    log_evs.append(le); data_fits.append(df); complexities.append(cp)
    aics.append(aic); bics.append(bic); log_liks.append(ll)
    m_N, S_N = bayesian_linear_regression(Phi, t, alpha, beta)
    y_plot = Phi_plot @ m_N
    y_std = np.sqrt(1.0 / beta + np.einsum("ij,jk,ik->i", Phi_plot, S_N, Phi_plot))
    fit_curves.append((y_plot, y_std))
    train_mses.append(float(np.mean((t - Phi @ m_N) ** 2)))
    test_mses.append(float(np.mean((t_test - Phi_test @ m_N) ** 2)))

import pandas as pd
pd.DataFrame({
    "M": orders,
    "ln p(D|M)": log_evs,
    "BIC": bics,
    "AIC": aics,
    "train MSE": train_mses,
    "test MSE": test_mses,
}).round(3)

## 6. Evidence peaks at an intermediate $M$

In [ ]:
best_M = int(np.argmax(log_evs))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(orders, log_evs, marker="o", color="#1B2D3D", linewidth=2)
ax.scatter([best_M], [log_evs[best_M]], color="#D4A24C", s=180, zorder=5, edgecolor="#1B2D3D")
ax.axvline(best_M, color="#D4A24C", linestyle="--", alpha=0.4)
ax.set_xlabel("Polynomial order M"); ax.set_ylabel("Log model evidence  ln p(D | M)")
ax.set_title(f"Model evidence peaks at M = {best_M}"); ax.grid(alpha=0.3)
plt.show()

print(f"Best by evidence: M = {best_M},  ln p(D|M) = {log_evs[best_M]:.2f}")
print(f"Best by test MSE: M = {int(np.argmin(test_mses))}")

## 7. Fit quality vs complexity penalty

Splitting the evidence into its two components (eq. 3.72 in Bishop, up to reparameterisation) shows how each term changes with $M$.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
xs_bar = np.arange(len(orders)); width = 0.35
ax.bar(xs_bar - width / 2, data_fits, width, color="#3D9B8F", label="Data-fit term")
ax.bar(xs_bar + width / 2, complexities, width, color="#D4A24C", label="Complexity penalty term")
ax.plot(xs_bar, log_evs, color="#1B2D3D", marker="o", linewidth=2, label="Sum = log evidence")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xticks(xs_bar); ax.set_xticklabels(orders)
ax.set_xlabel("Polynomial order M"); ax.set_ylabel("Contribution to ln p(D | M)")
ax.set_title("Evidence = fit quality + Occam penalty"); ax.legend(loc="lower left", fontsize=9); ax.grid(alpha=0.3, axis="y")
plt.show()

## 8. Evidence vs BIC vs AIC vs test MSE

BIC and AIC approximate the evidence well in the middle range of $M$, but they diverge catastrophically at the interpolation limit ($M = N - 1 = 9$) because $\sigma^2_{\text{ML}} \to 0$. The Bayesian evidence stays well-behaved.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
neg_2_log_ev = -2 * np.array(log_evs)
ax = axes[0]
ax.plot(orders, neg_2_log_ev, marker="o", color="#1B2D3D", linewidth=2, label="-2 ln p(D|M)  (Bayes evidence)")
ax.plot(orders, bics, marker="s", color="#3D9B8F", linewidth=2, label="BIC")
ax.plot(orders, aics, marker="^", color="#D4A24C", linewidth=2, label="AIC")
ax.set_xlabel("Polynomial order M"); ax.set_ylabel("Score (lower = better)")
ax.set_title("Evidence, BIC, AIC agree only approximately"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(orders, train_mses, marker="o", color="#1B2D3D", linewidth=2, label="Train MSE")
ax.plot(orders, test_mses, marker="s", color="#D4A24C", linewidth=2, label="Test MSE")
ax.set_xlabel("Polynomial order M"); ax.set_ylabel("MSE"); ax.set_yscale("log")
ax.set_title("Ground truth: held-out test error"); ax.legend(); ax.grid(alpha=0.3, which="both")
plt.show()

## 9. Occam's razor intuition (Bishop Fig 3.13)

A simple model concentrates predictive mass on a narrow range of datasets. A complex model spreads it thinly over many. For an observed dataset, the model of *just enough* complexity typically has the highest evidence.

In [ ]:
D = np.linspace(-3, 3, 800)
p1 = norm.pdf(D, 0.0, 0.5); p2 = norm.pdf(D, 0.6, 0.9); p3 = norm.pdf(D, 0.6, 2.0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(D, p1, color="#3D9B8F", linewidth=2.5, label="M\u2081 simple (narrow prior)")
ax.plot(D, p2, color="#D4A24C", linewidth=2.5, label="M\u2082 intermediate")
ax.plot(D, p3, color="#6B7280", linewidth=2.5, label="M\u2083 complex (broad prior)")
ax.axvline(0.6, color="#1B2D3D", linestyle="--", alpha=0.6, label="Observed data D\u2080")
ax.set_xlabel("Dataset space (schematic)"); ax.set_ylabel("p(D | M)")
ax.set_title("Why evidence favours intermediate complexity"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 10. Posterior predictive fits at selected $M$

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
for ax, M in zip(axes, [0, 1, 3, 9]):
    y_plot, y_std = fit_curves[M]
    ax.fill_between(x_plot, y_plot - y_std, y_plot + y_std, color="#3D9B8F", alpha=0.25)
    ax.plot(x_plot, true_function(x_plot), "k--", linewidth=1.2, label="truth")
    ax.plot(x_plot, y_plot, color="#1B2D3D", linewidth=2, label="posterior mean")
    ax.scatter(x, t, color="#D4A24C", edgecolor="#1B2D3D", zorder=5, s=45)
    ax.set_title(f"M = {M}   ln p(D|M) = {log_evs[M]:.2f}")
    ax.set_xlim(0, 1); ax.set_ylim(-1.8, 1.8); ax.set_xlabel("x")
axes[0].set_ylabel("t"); axes[0].legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

## Exercises

1. **Prior sensitivity.** Rerun the scan with $\alpha = 10^{-5}$ (very flat prior) and $\alpha = 10^{1}$ (tight prior). Where does the evidence peak in each case? Why does a very flat prior push the peak towards lower $M$?
2. **More data.** Repeat with $N = 100$ observations. Do BIC and evidence agree more closely? Does AIC still reward complex models more than BIC?
3. **Bayes factor.** Compute the Bayes factor $B_{ij} = p(\mathcal{D}|M_i)/p(\mathcal{D}|M_j)$ for $M_i = 3$ versus $M_j = 9$. Interpret it using Kass and Raftery's (1995) guidelines (e.g., $\log_{10} B > 2$ is decisive evidence).
4. **Evidence with unknown $\beta$.** Use the evidence approximation of §3.5.2: alternate between re-estimating $\alpha, \beta$ (via the fixed-point equations 3.92 and 3.95) and computing the evidence. Does the peak shift?